# B2-024 — Practice p26

*55 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** challenge · **Difficulty:** advanced · **Time budget:** 55 minutes  
**Concepts:** scientific-ml-inverse-problems

## Task

**Open-ended inverse-problem mini-competition.** Build any valid approach that locates the source from the eight sensor readings, beat the committed baseline on the locked test set within the budget, and write the rubric writeup.

**Fixed elements.**

* *Held-out set and score:* the locked `inverse` test rows, scored only by `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")`: mean source-position error (lower is better) with its bootstrap interval.
* *Baseline to beat:* `capstone_data.baseline_score("inverse").score` = `0.2137` at the frozen seed (grid Tikhonov). Any approach whose test score is **strictly lower**, within budget, is correct.
* *Budget:* create one `budget = capstone_data.StepBudget(2000)` before any fitting; wrap **every** optimizer with `budget.wrap(...)`; at most 2,000 optimizer steps in total (only non-iterative closed-form linear algebra, such as a single `torch.linalg.solve`, is free; any iterative solver — Adam, Gauss–Newton, L-BFGS, or a hand-written update loop — must run as a `budget.wrap(...)`-wrapped optimizer so every iteration counts as a step). Wall-clock time from the budget's creation to the end of the test call (`budget.elapsed()`) must be `< 15` seconds on a CPU.
* *Data:* `capstone_data.train_rows("inverse")`, `capstone_data.val_rows("inverse")` for selection only, the known forward model (Session 5), and optionally simulated pairs from `capstone_data.simulate("inverse", n, seed)` with `seed != SEED`.
* *Interface and seam:* your approach is an object with `fit(train_x, train_y)` and `predict(x)` (returning float `(N, 3)` = `(x, y, strength)`). `fit` must record `capstone_data.split_of("inverse", row)` for every row of `train_x`; no recorded value may be `"val"` or `"test"`.
* *Protocol:* Fit only on rows from `capstone_data.train_rows("inverse")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("inverse")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="inverse")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("inverse") == 0` immediately before that call and `== 1` after it.

**Writeup (required, four parts).**

* **Approach:** the model, its features, and why.
* **Alternatives considered:** at least two, each with the validation evidence that ruled it out.
* **Evaluation:** the validation protocol, the single final test score with its bootstrap interval, and the comparison with the baseline.
* **Limitations:** at least one failure mode and how you would test it.

**Required answer-check assertions.** Test score `<` the baseline score; `budget.used <= 2000`; `budget.elapsed() < 15` measured after the test call; no seam value in `{"val", "test"}`; the test-call protocol; the prediction shape `(200, 3)` on validation.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import time

MAX_STEPS = 2000
MAX_SECONDS = 15.0

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.